In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# ================================================================
# DEEPSEEK-CODER MALWARE CLASSIFICATION
# RESTORE SAVED ADAPTER + FULL VALIDATION EVALUATION
# ================================================================

import os
import glob
import re
import json
import shutil
import warnings
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel

warnings.filterwarnings("ignore")


# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

EXPECTED_VALIDATION_SIZE = 5898

BATCH_SIZE = 4
MAX_NEW_TOKENS = 20

ADAPTER_DIR = "/kaggle/working/deepseek_final_adapter"

OUTPUT_DIR = "/kaggle/working/deepseek_validation_results"

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = [
    "Benign",
    "RedLineStealer",
    "Downloader",
    "RAT",
    "BankingTrojan",
    "SnakeKeyLogger",
    "Spyware"
]

LABEL_MAP = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

ID_TO_LABEL = LABEL_MAP

print("=" * 70)
print("DEEPSEEK MALWARE VALIDATION")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Enable GPU in Kaggle settings.")

print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# 2. LOCATE DATASET
# ================================================================

csv_files = glob.glob(
    "/kaggle/input/**/*.csv",
    recursive=True
)

valid_csv_files = []

for path in csv_files:

    try:
        columns = pd.read_csv(path, nrows=0).columns

        if {"SHA256", "Type"}.issubset(columns):
            valid_csv_files.append(path)

    except Exception:
        continue

if not valid_csv_files:
    raise FileNotFoundError(
        "Malware_Dataset.csv was not found in Kaggle input."
    )

CSV_PATH = valid_csv_files[0]

print("\nDataset:", CSV_PATH)

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)

assert df["SHA256"].is_unique
assert df.isnull().sum().sum() == 0

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in ["SHA256", "Type"]
]

assert len(FEATURE_COLUMNS) == 318

assert all(
    pd.api.types.is_numeric_dtype(df[col])
    for col in FEATURE_COLUMNS
)

df["Type"] = pd.to_numeric(
    df["Type"],
    errors="raise"
).astype(int)

assert set(df["Type"].unique()) == set(LABEL_MAP.keys())

print("Dataset validation successful.")
print("Samples:", len(df))
print("PE features:", len(FEATURE_COLUMNS))


# ================================================================
# 3. RECONSTRUCT ORIGINAL STRATIFIED SPLIT
# ================================================================

development_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["Type"]
)

train_df, validation_df = train_test_split(
    development_df,
    test_size=0.25,
    random_state=SEED,
    stratify=development_df["Type"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

assert len(train_df) == 17693
assert len(validation_df) == EXPECTED_VALIDATION_SIZE
assert len(test_df) == 5898

assert set(train_df["SHA256"]).isdisjoint(
    validation_df["SHA256"]
)

assert set(train_df["SHA256"]).isdisjoint(
    test_df["SHA256"]
)

assert set(validation_df["SHA256"]).isdisjoint(
    test_df["SHA256"]
)

print("\nOriginal split reconstructed successfully.")

print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Testing:", len(test_df))

print("\nValidation class distribution:")
print(validation_df["Type"].value_counts().sort_index())


# ================================================================
# 4. LOCATE AND RESTORE SAVED ADAPTER FILES
# ================================================================

all_files = glob.glob(
    "/kaggle/input/**/*",
    recursive=True
)

def find_file(prefix, extension):

    matches = [
        path for path in all_files
        if os.path.isfile(path)
        and os.path.basename(path).startswith(prefix)
        and path.endswith(extension)
    ]

    if not matches:
        raise FileNotFoundError(
            f"Could not find {prefix}*{extension} in Kaggle input."
        )

    if len(matches) > 1:
        print(f"Multiple matches for {prefix}; using:", matches[0])

    return matches[0]


adapter_config_source = find_file(
    "adapter_config",
    ".json"
)

adapter_weights_source = find_file(
    "adapter_model",
    ".safetensors"
)

tokenizer_source = find_file(
    "tokenizer",
    ".json"
)

tokenizer_config_source = find_file(
    "tokenizer_config",
    ".json"
)

files_to_restore = {
    adapter_config_source: "adapter_config.json",
    adapter_weights_source: "adapter_model.safetensors",
    tokenizer_source: "tokenizer.json",
    tokenizer_config_source: "tokenizer_config.json"
}

print("\nRestoring adapter files...")

for source, destination_name in files_to_restore.items():

    destination = os.path.join(
        ADAPTER_DIR,
        destination_name
    )

    shutil.copy2(source, destination)

    print(
        destination_name,
        "|",
        round(os.path.getsize(destination) / (1024 ** 2), 2),
        "MB"
    )

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json"
]

for filename in required_files:

    assert os.path.exists(
        os.path.join(ADAPTER_DIR, filename)
    ), f"Missing adapter file: {filename}"

print("\nAdapter files restored successfully.")


# ================================================================
# 5. LOAD SAVED TOKENIZER
# ================================================================

print("\nLoading saved tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR,
    trust_remote_code=True,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Tokenizer loaded.")
print("EOS token:", tokenizer.eos_token)
print("EOS ID:", tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token)


# ================================================================
# 6. RECREATE ORIGINAL FEATURE GROUPS
# ================================================================

API_FEATURES = FEATURE_COLUMNS[:200]

DLL_FEATURES = FEATURE_COLUMNS[200:227]

HEADER_FEATURES = FEATURE_COLUMNS[227:273]

SECTION_FEATURES = FEATURE_COLUMNS[273:318]

assert len(API_FEATURES) == 200
assert len(DLL_FEATURES) == 27
assert len(HEADER_FEATURES) == 46
assert len(SECTION_FEATURES) == 45


# ================================================================
# 7. ORIGINAL TRAINING PROMPT TEMPLATE
# ================================================================

SYSTEM_PROMPT = """You are a Windows PE malware classification assistant.

Classify the executable using only its provided static PE features.

Instructions:
1. Analyze all provided feature values.
2. Zero values are meaningful and must not be ignored.
3. Do not invent missing information.
4. Do not infer unsupported runtime behavior.
5. Select exactly one malware family.

Available classes:
Benign, RedLineStealer, Downloader, RAT,
BankingTrojan, SnakeKeyLogger, Spyware.
"""


def format_features(row, columns):

    return "\n".join(
        f"{col}={row[col]}"
        for col in columns
    )


def create_prompt(row):

    return f"""### SYSTEM:
{SYSTEM_PROMPT}

### USER:
Analyze the following Windows PE static feature profile.

<API_FUNCTIONS>
{format_features(row, API_FEATURES)}
</API_FUNCTIONS>

<DLL_IMPORTS>
{format_features(row, DLL_FEATURES)}
</DLL_IMPORTS>

<PE_HEADER>
{format_features(row, HEADER_FEATURES)}
</PE_HEADER>

<PE_SECTIONS>
{format_features(row, SECTION_FEATURES)}
</PE_SECTIONS>

Classify this executable into exactly one of the seven malware families.

### ASSISTANT:
"""


# ================================================================
# 8. LOAD BASE MODEL IN 4-BIT NF4
# ================================================================

print("\nLoading DeepSeek-Coder base model...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

print("Base model loaded.")

# Load the saved trained LoRA adapter.

print("\nLoading trained LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False
)

model.eval()

print("Trained adapter loaded successfully.")

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)


# ================================================================
# 9. PREPARE VALIDATION DATA
# ================================================================

validation_prompts = []

validation_labels = []

validation_hashes = []

for _, row in validation_df.iterrows():

    prompt = create_prompt(row)

    encoded = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=False
    )

    input_ids = encoded["input_ids"]

    if len(input_ids) > 4096:
        raise ValueError(
            f"Validation prompt exceeds 4096 tokens: {len(input_ids)}"
        )

    validation_prompts.append(input_ids)

    validation_labels.append(
        LABEL_MAP[int(row["Type"])]
    )

    validation_hashes.append(
        str(row["SHA256"])
    )

assert len(validation_prompts) == EXPECTED_VALIDATION_SIZE

print("\nValidation prompts prepared.")
print("Total prompts:", len(validation_prompts))
print(
    "Maximum prompt length:",
    max(len(x) for x in validation_prompts)
)


# ================================================================
# 10. PREDICTION PARSER
# ================================================================

# Longest class names first to avoid partial matching.

SORTED_CLASSES = sorted(
    CLASS_NAMES,
    key=len,
    reverse=True
)

CLASS_PATTERN = re.compile(
    r"\b(" + "|".join(
        re.escape(name)
        for name in SORTED_CLASSES
    ) + r")\b",
    flags=re.IGNORECASE
)


def extract_prediction(text):

    match = CLASS_PATTERN.search(text)

    if match is None:
        return "INVALID"

    predicted = match.group(1)

    for class_name in CLASS_NAMES:

        if predicted.lower() == class_name.lower():
            return class_name

    return "INVALID"


# ================================================================
# 11. RUN FULL VALIDATION INFERENCE
# ================================================================

print("\nStarting validation inference...")

predictions = []

model_device = next(model.parameters()).device

total_batches = (
    len(validation_prompts) + BATCH_SIZE - 1
) // BATCH_SIZE

for batch_start in tqdm(
    range(0, len(validation_prompts), BATCH_SIZE),
    total=total_batches,
    desc="Validation"
):

    batch_prompts = validation_prompts[
        batch_start:batch_start + BATCH_SIZE
    ]

    batch_max_length = max(
        len(x) for x in batch_prompts
    )

    padded_ids = []
    attention_masks = []

    for ids in batch_prompts:

        padding_length = batch_max_length - len(ids)

        padded_ids.append(
            [tokenizer.pad_token_id] * padding_length + ids
        )

        attention_masks.append(
            [0] * padding_length + [1] * len(ids)
        )

    input_ids = torch.tensor(
        padded_ids,
        dtype=torch.long,
        device=model_device
    )

    attention_mask = torch.tensor(
        attention_masks,
        dtype=torch.long,
        device=model_device
    )

    with torch.inference_mode():

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    # Decode only newly generated tokens.

    generated_tokens = outputs[
        :, input_ids.shape[1]:
    ]

    decoded_outputs = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    for generated_text in decoded_outputs:

        prediction = extract_prediction(
            generated_text
        )

        predictions.append(prediction)

    del input_ids
    del attention_mask
    del outputs
    del generated_tokens

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Save intermediate progress every 100 batches.

    current_batch = (
        batch_start // BATCH_SIZE
    ) + 1

    if current_batch % 100 == 0:

        partial_df = pd.DataFrame({
            "SHA256": validation_hashes[:len(predictions)],
            "Actual": validation_labels[:len(predictions)],
            "Predicted": predictions
        })

        partial_df.to_csv(
            os.path.join(
                OUTPUT_DIR,
                "validation_predictions_partial.csv"
            ),
            index=False
        )

        print(
            f"\nProgress saved: {len(predictions)}/"
            f"{EXPECTED_VALIDATION_SIZE}"
        )


# ================================================================
# 12. VERIFY PREDICTION COUNT
# ================================================================

assert len(predictions) == EXPECTED_VALIDATION_SIZE, (
    f"Expected {EXPECTED_VALIDATION_SIZE} predictions, "
    f"received {len(predictions)}"
)

print("\nInference completed successfully.")

actual = validation_labels

invalid_count = predictions.count("INVALID")

print("Total evaluated:", len(predictions))
print("Invalid predictions:", invalid_count)


# ================================================================
# 13. CALCULATE EVALUATION METRICS
# ================================================================

accuracy = accuracy_score(
    actual,
    predictions
)

precision_macro = precision_score(
    actual,
    predictions,
    labels=CLASS_NAMES,
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    actual,
    predictions,
    labels=CLASS_NAMES,
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    actual,
    predictions,
    labels=CLASS_NAMES,
    average="macro",
    zero_division=0
)

f1_weighted = f1_score(
    actual,
    predictions,
    labels=CLASS_NAMES,
    average="weighted",
    zero_division=0
)

balanced_accuracy = balanced_accuracy_score(
    actual,
    predictions
)


# ================================================================
# 14. PRINT FINAL METRICS
# ================================================================

print("\n" + "=" * 70)
print("DEEPSEEK-CODER VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy:           {accuracy:.4f}")
print(f"Macro Precision:    {precision_macro:.4f}")
print(f"Macro Recall:       {recall_macro:.4f}")
print(f"Macro F1:           {f1_macro:.4f}")
print(f"Weighted F1:        {f1_weighted:.4f}")
print(f"Balanced Accuracy:  {balanced_accuracy:.4f}")

print(f"Invalid Predictions: {invalid_count}")

print("\nClassification Report:\n")

print(
    classification_report(
        actual,
        predictions,
        labels=CLASS_NAMES,
        zero_division=0,
        digits=4
    )
)


# ================================================================
# 15. CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    actual,
    predictions,
    labels=CLASS_NAMES
)

confusion_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

print("\nConfusion Matrix:\n")

print(confusion_df)


# ================================================================
# 16. SAVE VALIDATION RESULTS
# ================================================================

results_df = pd.DataFrame({
    "SHA256": validation_hashes,
    "Actual": actual,
    "Predicted": predictions
})

results_path = os.path.join(
    OUTPUT_DIR,
    "deepseek_validation_predictions.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

confusion_path = os.path.join(
    OUTPUT_DIR,
    "deepseek_validation_confusion_matrix.csv"
)

confusion_df.to_csv(confusion_path)

metrics = {
    "model": MODEL_NAME,
    "adapter": ADAPTER_DIR,
    "validation_samples": len(actual),
    "accuracy": float(accuracy),
    "macro_precision": float(precision_macro),
    "macro_recall": float(recall_macro),
    "macro_f1": float(f1_macro),
    "weighted_f1": float(f1_weighted),
    "balanced_accuracy": float(balanced_accuracy),
    "invalid_predictions": int(invalid_count)
}

metrics_path = os.path.join(
    OUTPUT_DIR,
    "deepseek_validation_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)


# ================================================================
# 17. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("VALIDATION EVALUATION COMPLETED")
print("=" * 70)

print("\nSaved files:")

print(results_path)
print(confusion_path)
print(metrics_path)

print("\nFinal result:")
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Macro F1: {f1_macro * 100:.2f}%")
print(f"Weighted F1: {f1_weighted * 100:.2f}%")

print("\nThe test dataset has NOT been evaluated.")
print("The trained DeepSeek adapter has NOT been modified.")

print("\nAll done.")

DEEPSEEK MALWARE VALIDATION
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

Dataset: /kaggle/input/datasets/mohdtahasayed2904/dataaset-12342w462356/Malware_Dataset.csv
Dataset shape: (29489, 320)
Dataset validation successful.
Samples: 29489
PE features: 318

Original split reconstructed successfully.
Training: 17693
Validation: 5898
Testing: 5898

Validation class distribution:
Type
0     375
1    1004
2     929
3     991
4    1015
5     844
6     740
Name: count, dtype: int64
Multiple matches for tokenizer; using: /kaggle/input/datasets/mohdtahasayed2904/validation/tokenizer (3).json

Restoring adapter files...
adapter_config.json | 0.0 MB
adapter_model.safetensors | 57.23 MB
tokenizer.json | 2.18 MB
tokenizer_config.json | 0.0 MB

Adapter files restored successfully.

Loading saved tokenizer...
Tokenizer loaded.
EOS token: <｜end▁of▁sentence｜>
EOS ID: 32014
PAD token: <｜end▁of▁sentence｜>

Loading DeepSeek-Coder base model...


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.69GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Base model loaded.

Loading trained LoRA adapter...
Trained adapter loaded successfully.
Trainable parameters: 0

Validation prompts prepared.
Total prompts: 5898
Maximum prompt length: 2946

Starting validation inference...


Validation:   0%|          | 0/1475 [00:00<?, ?it/s]


Progress saved: 400/5898

Progress saved: 800/5898

Progress saved: 1200/5898

Progress saved: 1600/5898

Progress saved: 2000/5898

Progress saved: 2400/5898

Progress saved: 2800/5898

Progress saved: 3200/5898

Progress saved: 3600/5898

Progress saved: 4000/5898

Progress saved: 4400/5898

Progress saved: 4800/5898

Progress saved: 5200/5898

Progress saved: 5600/5898

Inference completed successfully.
Total evaluated: 5898
Invalid predictions: 0

DEEPSEEK-CODER VALIDATION RESULTS
Accuracy:           0.7404
Macro Precision:    0.7461
Macro Recall:       0.7170
Macro F1:           0.7275
Weighted F1:        0.7462
Balanced Accuracy:  0.7170
Invalid Predictions: 0

Classification Report:

                precision    recall  f1-score   support

        Benign     0.7786    0.5627    0.6533       375
RedLineStealer     0.9116    0.8118    0.8588      1004
    Downloader     0.9811    0.9505    0.9656       929
           RAT     0.7490    0.7407    0.7448       991
 BankingTrojan    